<h2 style='color:purple' align='center'>Naive Bayes Tutorial Part 1: Predicting survival from titanic crash</h2>

In [1]:
import pandas as pd
df=pd.read_csv("titanic.csv")
df.head()

,PassengerId,Name,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female,35.0,1,0,113803,53.1000,C123,S,1
4,5,"Allen, Mr. William Henry",3,male,35.0,0,0,373450,8.0500,NaN,S,0


In [2]:
df.columns

Index(['PassengerId', 'Name', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'Survived'],
      dtype='object')

In [3]:
df.drop(["PassengerId","Name","SibSp","Parch","Ticket","Cabin","Embarked"],axis="columns",inplace=True)
df.head()

,Pclass,Sex,Age,Fare,Survived
0,3,male,22.0,7.2500,0
1,1,female,38.0,71.2833,1
2,3,female,26.0,7.9250,1
3,1,female,35.0,53.1000,1
4,3,male,35.0,8.0500,0


In [4]:
target=df["Survived"]
inputs=df.drop(["Survived"],axis="columns")

In [5]:
#inputs.Sex = inputs.Sex.map({'male': 1, 'female': 2})

In [5]:
dummies=pd.get_dummies(inputs["Sex"])
dummies.head()

,female,male
0,False,True
1,True,False
2,True,False
3,True,False
4,False,True


In [6]:
inputs=pd.concat([inputs,dummies],axis="columns")
inputs.head()

,Pclass,Sex,Age,Fare,female,male
0,3,male,22.0,7.2500,False,True
1,1,female,38.0,71.2833,True,False
2,3,female,26.0,7.9250,True,False
3,1,female,35.0,53.1000,True,False
4,3,male,35.0,8.0500,False,True


In [7]:
inputs.drop(["Sex","male"],axis="columns",inplace=True)

**I am dropping male column as well because of dummy variable trap theory. One column is enough to repressent male vs female**

In [8]:
inputs.head()

,Pclass,Age,Fare,female
0,3,22.0,7.2500,False
1,1,38.0,71.2833,True
2,3,26.0,7.9250,True
3,1,35.0,53.1000,True
4,3,35.0,8.0500,False


In [9]:
inputs.columns[inputs.isna().any()]

Index(['Age'], dtype='object')

In [11]:
inputs.Age[:10]

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5     NaN
6    54.0
7     2.0
8    27.0
9    14.0
Name: Age, dtype: float64

In [12]:
inputs["Age"].fillna(inputs["Age"].mean(),inplace=True)

/var/folders/48/m5h2lpdd1nlc3bhw471fn9c00000gn/T/ipykernel_30058/827987996.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  inputs["Age"].fillna(inputs["Age"].mean(),inplace=True)


In [14]:
inputs.Age[:10]

0    22.000000
1    38.000000
2    26.000000
3    35.000000
4    35.000000
5    29.699118
6    54.000000
7     2.000000
8    27.000000
9    14.000000
Name: Age, dtype: float64

In [15]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(inputs,target,test_size=0.2,random_state=10)

In [16]:
len(X_train)

712

In [17]:
len(X_test)

179

In [18]:
len(inputs)

891

In [19]:
from sklearn.naive_bayes import GaussianNB
model=GaussianNB()

In [20]:
model.fit(X_train,y_train)

,priors,None
,var_smoothing,1e-09


In [21]:
model.score(X_test,y_test)

0.8212290502793296

In [22]:
model.predict(X_test[:10])

array([0, 0, 0, 1, 1, 0, 0, 0, 0, 0])

In [23]:
y_test[:10]

590    0
131    0
628    0
195    1
230    1
646    0
75     0
586    0
569    1
287    0
Name: Survived, dtype: int64

In [24]:
model.predict_proba(X_test[:10])

array([[9.62058761e-01, 3.79412387e-02],
       [9.56142276e-01, 4.38577240e-02],
       [9.60258068e-01, 3.97419319e-02],
       [1.51442683e-04, 9.99848557e-01],
       [2.19546518e-02, 9.78045348e-01],
       [9.55492143e-01, 4.45078574e-02],
       [9.59706672e-01, 4.02933276e-02],
       [9.17654736e-01, 8.23452639e-02],
       [9.62036278e-01, 3.79637224e-02],
       [9.57930671e-01, 4.20693286e-02]])

**Calculate the score using cross validation**

In [34]:
from sklearn.model_selection import cross_val_score
cross_val_score(GaussianNB(),X_train, y_train, cv=5)

array([0.75396825, 0.784     , 0.76612903, 0.82258065, 0.77419355])